In [15]:
import os
import pandas as pd
import numpy as np
from scipy.linalg import block_diag
from category_encoders import BinaryEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

In [16]:
data_root_dir = os.path.abspath('../data/TabFormer/') 
tabformer_raw_file_path = os.path.join(
        data_root_dir, "raw", "card_transaction.v1.csv"
    )

In [22]:
data = pd.read_csv(tabformer_raw_file_path, nrows=1000000)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   User            int64  
 1   Card            int64  
 2   Year            int64  
 3   Month           int64  
 4   Day             int64  
 5   Time            object 
 6   Amount          object 
 7   Use Chip        object 
 8   Merchant Name   int64  
 9   Merchant City   object 
 10  Merchant State  object 
 11  Zip             float64
 12  MCC             int64  
 13  Errors?         object 
 14  Is Fraud?       object 
dtypes: float64(1), int64(7), object(7)
memory usage: 2.7+ GB


In [23]:
data.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


In [24]:
data['Use Chip'].unique()

array(['Swipe Transaction', 'Online Transaction', 'Chip Transaction'],
      dtype=object)

In [25]:
data['Errors?'].unique()

array([nan, 'Technical Glitch,', 'Insufficient Balance,', 'Bad PIN,',
       'Bad PIN,Insufficient Balance,', 'Bad Expiration,',
       'Bad PIN,Technical Glitch,', 'Bad Card Number,', 'Bad CVV,',
       'Bad Zipcode,', 'Insufficient Balance,Technical Glitch,',
       'Bad Card Number,Insufficient Balance,',
       'Bad Card Number,Bad CVV,', 'Bad CVV,Insufficient Balance,',
       'Bad Card Number,Bad Expiration,', 'Bad Expiration,Bad CVV,',
       'Bad Expiration,Insufficient Balance,',
       'Bad Expiration,Technical Glitch,',
       'Bad Card Number,Bad Expiration,Technical Glitch,',
       'Bad CVV,Technical Glitch,', 'Bad Card Number,Technical Glitch,',
       'Bad Zipcode,Insufficient Balance,',
       'Bad Zipcode,Technical Glitch,',
       'Bad Card Number,Bad Expiration,Insufficient Balance,'],
      dtype=object)

In [26]:
# Apply same cleaning as preprocess_TabFormer.py
print("\nApplying data cleaning...")

# Rename columns
data = data.rename(columns={
    "Merchant Name": "Merchant",
    "Merchant State": "State",
    "Merchant City": "City",
    "Errors?": "Errors",
    "Use Chip": "Chip",
    "Is Fraud?": "Fraud",
})

# Handle missing values
data['State'] = data['State'].fillna('XX')
data['Errors'] = data['Errors'].fillna('XX')
data['Zip'] = data['Zip'].fillna(0)

# Clean Amount
data['Amount'] = data['Amount'].str.replace('$', '').astype('float')

# Convert Fraud
data['Fraud'] = data['Fraud'].map({'No': 0, 'Yes': 1}).astype('int8')

# Remove commas from Errors
data['Errors'] = data['Errors'].str.replace(',', '')

# Convert Time to minutes
time_split = data['Time'].str.split(':', expand=True)
data['Time'] = (time_split[0].astype('int32') * 60 + time_split[1].astype('int32'))

# Convert Merchant to string
data['Merchant'] = data['Merchant'].astype('str')

# Combine User and Card
max_cards = len(data['Card'].unique())
print(max_cards)
data['Card'] = data['User'] * max_cards + data['Card']
data['Card'] = data['Card'].astype('int')


Applying data cleaning...
9


In [21]:
# Filter to training data (Year < 2018)
training_idx = data['Year'] < 2018
validation_idx = data['Year'] == 2018
test_idx = data['Year'] > 2018

training_data = data[data['Year'] < 2018].copy()

print(f"Training data: {len(training_data)} rows")

Training data: 86569 rows


In [14]:
#  MCC (Merchant Category Code), is a four-digit number used to classify businesses based on the goods or services they provide
training_data.head()

,User,Card,Year,Month,Day,Time,Amount,Chip,Merchant,City,State,Zip,MCC,Errors,Fraud
0,0,0,2002,9,1,381,134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,XX,0
1,0,0,2002,9,1,402,38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,XX,0
2,0,0,2002,9,2,382,120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,XX,0
3,0,0,2002,9,2,1065,128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,XX,0
4,0,0,2002,9,3,383,104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,XX,0


In [28]:
training_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 86569 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   User      86569 non-null  int64  
 1   Card      86569 non-null  int64  
 2   Year      86569 non-null  int64  
 3   Month     86569 non-null  int64  
 4   Day       86569 non-null  int64  
 5   Time      86569 non-null  int32  
 6   Amount    86569 non-null  float64
 7   Chip      86569 non-null  object 
 8   Merchant  86569 non-null  object 
 9   City      86569 non-null  object 
 10  State     86569 non-null  object 
 11  Zip       86569 non-null  float64
 12  MCC       86569 non-null  int64  
 13  Errors    86569 non-null  object 
 14  Fraud     86569 non-null  int8   
dtypes: float64(2), int32(1), int64(6), int8(1), object(5)
memory usage: 9.7+ MB


In [8]:
# ========================================
# 1. Create and fit ID transformer
# ========================================
print("\n" + "=" * 60)
print("Creating ID Transformer (Card, Merchant, MCC)")
print("=" * 60)

MERCHANT_AND_USER_COLS = ["Merchant", "Card", "MCC"]

# Create sample data for fitting
nr_unique_card = training_data['Card'].unique().shape[0]
nr_unique_merchant = training_data['Merchant'].unique().shape[0]
nr_unique_mcc = training_data['MCC'].unique().shape[0]
nr_elements = max(nr_unique_merchant, nr_unique_card)

data_ids = pd.DataFrame()
data_ids['Card'] = [training_data['Card'].iloc[0]] * nr_elements
data_ids['Merchant'] = [training_data['Merchant'].iloc[0]] * nr_elements
data_ids['MCC'] = [training_data['MCC'].iloc[0]] * nr_elements

data_ids.loc[np.arange(nr_unique_card), 'Card'] = training_data['Card'].unique()
data_ids.loc[np.arange(nr_unique_merchant), 'Merchant'] = training_data['Merchant'].unique()
data_ids.loc[np.arange(nr_unique_mcc), 'MCC'] = training_data['MCC'].unique()

data_ids = data_ids[MERCHANT_AND_USER_COLS].astype('category')


Creating ID Transformer (Card, Merchant, MCC)


In [36]:
data_ids.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1972 entries, 0 to 1971
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   Merchant  1972 non-null   category
 1   Card      1972 non-null   category
 2   MCC       1972 non-null   category
dtypes: category(3)
memory usage: 93.4 KB


In [15]:
data_ids.head()

,Merchant,Card,MCC
0,3527213246127876953,0,5300
1,-727612092139916043,1,5411
2,3414527459579106770,2,5651
3,5817218446178736267,3,5912
4,-7146670748125200898,4,5970


In [11]:
# Create ID transformer
id_bin_encoder = Pipeline(
    steps=[
        ("binary", BinaryEncoder(handle_missing="value", handle_unknown="value"))
    ]
)

id_transformer = ColumnTransformer(
    transformers=[
        ("binary", id_bin_encoder, MERCHANT_AND_USER_COLS),
    ],
    remainder="passthrough",
)

# Fit ID transformer
pd.set_option("future.no_silent_downcasting", True)
id_transformer = id_transformer.fit(data_ids)

print(f"✓ ID transformer fitted")
print(f"  Input columns: {MERCHANT_AND_USER_COLS}")
print(f"  Output columns: {len(id_transformer.get_feature_names_out())}")

✓ ID transformer fitted
  Input columns: ['Merchant', 'Card', 'MCC']
  Output columns: 23


In [16]:
# ========================================
# 2. Create and fit feature transformer
# ========================================
print("\n" + "=" * 60)
print("Creating Feature Transformer")
print("=" * 60)

# Define predictor columns
numerical_predictors = ['Amount']
nominal_predictors = ['Errors', 'Card', 'Chip', 'City', 'Zip', 'MCC', 'Merchant']

# Remove ID columns from nominal predictors (they're handled separately)
nominal_predictors = [col for col in nominal_predictors if col not in MERCHANT_AND_USER_COLS]

predictor_columns = numerical_predictors + nominal_predictors

# Prepare training subset
pdf_training = training_data[predictor_columns + ['Fraud']].copy()
pdf_training[nominal_predictors] = pdf_training[nominal_predictors].astype('category')

# Determine encoding strategy
columns_for_binary_encoding = []
columns_for_one_hot_encoding = []

for col in nominal_predictors:
    if len(training_data[col].unique()) <= 8:
        columns_for_one_hot_encoding.append(col)
    else:
        columns_for_binary_encoding.append(col)

print(f"  Binary encoding: {columns_for_binary_encoding}")
print(f"  One-hot encoding: {columns_for_one_hot_encoding}")


Creating Feature Transformer
  Binary encoding: ['Errors', 'City', 'Zip']
  One-hot encoding: ['Chip']


In [17]:
# Create encoders
bin_encoder = Pipeline(
    steps=[
        ("binary", BinaryEncoder(handle_missing="value", handle_unknown="value"))
    ]
)
one_hot_encoder = Pipeline(steps=[("onehot", OneHotEncoder())])

robust_scaler = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("robust", RobustScaler()),
    ],
)

# Create feature transformer
transformer = ColumnTransformer(
    transformers=[
        ("binary", bin_encoder, columns_for_binary_encoding),
        ("onehot", one_hot_encoder, columns_for_one_hot_encoding),
        ("robust", robust_scaler, ['Amount']),
    ],
    remainder="passthrough",
)

# Fit feature transformer
transformer = transformer.fit(pdf_training[predictor_columns])

print(f"✓ Feature transformer fitted")
print(f"  Input columns: {len(predictor_columns)}")
print(f"  Output columns: {len(transformer.get_feature_names_out())}")

✓ Feature transformer fitted
  Input columns: 5
  Output columns: 30


In [24]:
# transformed column names
columns_of_transformed_data = list(
    map(
        lambda name: name.split("__")[1],
        list(transformer.get_feature_names_out(predictor_columns)),
    )
)

# transformed column names
columns_of_transformed_id_data = list(
    map(
        lambda name: name.split("__")[1],
        list(id_transformer.get_feature_names_out(MERCHANT_AND_USER_COLS)),
    )
)

columns_of_transformed_data, columns_of_transformed_id_data

(['Errors_0',
  'Errors_1',
  'Errors_2',
  'Errors_3',
  'City_0',
  'City_1',
  'City_2',
  'City_3',
  'City_4',
  'City_5',
  'City_6',
  'City_7',
  'City_8',
  'City_9',
  'City_10',
  'Zip_0',
  'Zip_1',
  'Zip_2',
  'Zip_3',
  'Zip_4',
  'Zip_5',
  'Zip_6',
  'Zip_7',
  'Zip_8',
  'Zip_9',
  'Zip_10',
  'Chip_Chip Transaction',
  'Chip_Online Transaction',
  'Chip_Swipe Transaction',
  'Amount'],
 ['Merchant_0',
  'Merchant_1',
  'Merchant_2',
  'Merchant_3',
  'Merchant_4',
  'Merchant_5',
  'Merchant_6',
  'Merchant_7',
  'Merchant_8',
  'Merchant_9',
  'Merchant_10',
  'Card_0',
  'Card_1',
  'Card_2',
  'Card_3',
  'Card_4',
  'MCC_0',
  'MCC_1',
  'MCC_2',
  'MCC_3',
  'MCC_4',
  'MCC_5',
  'MCC_6'])

In [25]:
# data type of transformed columns
type_mapping = {}
for col in columns_of_transformed_data:
    if col.split("_")[0] in nominal_predictors:
        type_mapping[col] = "int8"
    elif col in numerical_predictors:
        type_mapping[col] = "float"
    elif col in target_column:
        type_mapping[col] = data.dtypes.to_dict()[col]

# transform training data
preprocessed_training_data = transformer.transform(pdf_training[predictor_columns])

# Convert transformed data to panda DataFrame
preprocessed_training_data = pd.DataFrame(
    preprocessed_training_data, columns=columns_of_transformed_data
)

# Transform test data using the transformer fitted on training data
# pdf_test = data[test_idx][predictor_columns + target_column]
# pdf_test[nominal_predictors] = pdf_test[nominal_predictors].astype("category")

# preprocessed_test_data = transformer.transform(pdf_test[predictor_columns])
# preprocessed_test_data = pd.DataFrame(
#     preprocessed_test_data, columns=columns_of_transformed_data
# )

# # Transform validation data using the transformer fitted on training data
# pdf_validation = data[validation_idx][predictor_columns + target_column]
# pdf_validation[nominal_predictors] = pdf_validation[nominal_predictors].astype(
#     "category"
# )

# preprocessed_validation_data = transformer.transform(
#     pdf_validation[predictor_columns]
# )
# preprocessed_validation_data = pd.DataFrame(
#     preprocessed_validation_data, columns=columns_of_transformed_data
# )

preprocessed_id_data_train = pd.DataFrame(
    id_transformer.transform(data[training_idx][MERCHANT_AND_USER_COLS]),
    columns=columns_of_transformed_id_data,
)
preprocessed_training_data = pd.concat(
    [preprocessed_training_data, preprocessed_id_data_train], axis=1
)

In [44]:
preprocessed_training_data.filter(regex='^Chip')

,Chip_Chip Transaction,Chip_Online Transaction,Chip_Swipe Transaction
0,0.0,0.0,1.0
1,0.0,0.0,1.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0
...,...,...,...
86564,0.0,1.0,0.0
86565,0.0,0.0,1.0
86566,0.0,0.0,1.0
86567,0.0,0.0,1.0


In [45]:
COL_USER = "User"
COL_CARD = "Card"
COL_AMOUNT = "Amount"
COL_MCC = "MCC"
COL_TIME = "Time"
COL_DAY = "Day"
COL_MONTH = "Month"
COL_YEAR = "Year"

COL_MERCHANT = "Merchant"
COL_STATE = "State"
COL_CITY = "City"
COL_ZIP = "Zip"
COL_ERROR = "Errors"
COL_CHIP = "Chip"
COL_FRAUD = "Fraud"
COL_TRANSACTION_ID = "Tx_ID"
COL_MERCHANT_ID = "Merchant_ID"
COL_USER_ID = "User_ID"

COL_GRAPH_SRC = "src"
COL_GRAPH_DST = "dst"
COL_GRAPH_WEIGHT = "wgt"
MERCHANT_AND_USER_COLS = [COL_MERCHANT, COL_CARD, COL_MCC]

In [50]:
# ### GNN Data

# #### Setting Vertex IDs
# In order to create a graph, the different vertices need to be assigned unique vertex IDs. Additionally, the IDs needs to be consecutive and positive.
#
# There are three nodes groups here: Transactions, Users, and Merchants.
#
# This IDs are not used in training, just used for graph processing.

# Use the same training data as used for XGBoost

data_all = training_data.copy()
data_gnn = data_all #pd.concat([data[training_idx], data[validation_idx]])
data_gnn.reset_index(inplace=True, drop=True)

# The number of transaction is the same as the size of the list, and hence the index value
data_gnn[COL_TRANSACTION_ID] = data_gnn.index

merchant_name_to_id = dict(
    zip(data_gnn[COL_MERCHANT].unique(), np.arange(len(data_gnn[COL_MERCHANT].unique())))
)

data_gnn[COL_MERCHANT_ID] = data_gnn[COL_MERCHANT].map(merchant_name_to_id)

# ##### NOTE: the 'User' and 'Card' columns of the original data were used to crate updated 'Card' column
# * You can use user or card as nodes

id_to_consecutive_id = dict(
    zip(data_gnn[COL_CARD].unique(), np.arange(len(data_gnn[COL_CARD].unique())))
)

# Convert Card to consecutive IDs
data_gnn[COL_USER_ID] = data_gnn[COL_CARD].map(id_to_consecutive_id)

NR_USERS = data_gnn[COL_USER_ID].max() + 1
NR_MXS = data_gnn[COL_MERCHANT_ID].max() + 1
NR_TXS = data_gnn[COL_TRANSACTION_ID].max() + 1

# Check the the transaction, merchant and user ids are consecutive
id_range = data_gnn[COL_TRANSACTION_ID].min(), data_gnn[COL_TRANSACTION_ID].max()
print(f"Transaction ID range {id_range}")
id_range = data_gnn[COL_MERCHANT_ID].min(), data_gnn[COL_MERCHANT_ID].max()
print(f"Merchant ID range {id_range}")
id_range = data_gnn[COL_USER_ID].min(), data_gnn[COL_USER_ID].max()
print(f"User ID range {id_range}")

Transaction ID range (0, 86568)
Merchant ID range (0, 1971)
User ID range (0, 19)


In [52]:
data_gnn.head()

,User,Card,Year,Month,Day,Time,Amount,Chip,Merchant,City,State,Zip,MCC,Errors,Fraud,Tx_ID,Merchant_ID,User_ID
0,0,0,2002,9,1,381,134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,XX,0,0,0,0
1,0,0,2002,9,1,402,38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,XX,0,1,1,0
2,0,0,2002,9,2,382,120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,XX,0,2,1,0
3,0,0,2002,9,2,1065,128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,XX,0,3,2,0
4,0,0,2002,9,3,383,104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,XX,0,4,3,0


In [55]:
NR_USERS, NR_MXS, NR_TXS

(20, 1972, 86569)

In [57]:
# #### Create Edge in COO format

# User to Transactions
U_2_T = pd.DataFrame()
U_2_T[COL_GRAPH_SRC] = data_gnn[COL_USER_ID]
U_2_T[COL_GRAPH_DST] = data_gnn[COL_TRANSACTION_ID] + NR_USERS + NR_MXS

# Transactions to Merchants
T_2_M = pd.DataFrame()
T_2_M[COL_GRAPH_SRC] = data_gnn[COL_TRANSACTION_ID] + NR_USERS + NR_MXS
T_2_M[COL_GRAPH_DST] = data_gnn[COL_MERCHANT_ID] + NR_USERS

# Transactions to Users
T_2_U = pd.DataFrame()
T_2_U[COL_GRAPH_SRC] = data_gnn[COL_TRANSACTION_ID] + NR_USERS + NR_MXS
T_2_U[COL_GRAPH_DST] = data_gnn[COL_USER_ID]

# Merchants to Transactions
M_2_T = pd.DataFrame()
M_2_T[COL_GRAPH_SRC] = data_gnn[COL_MERCHANT_ID] + NR_USERS
M_2_T[COL_GRAPH_DST] = data_gnn[COL_TRANSACTION_ID] + NR_USERS + NR_MXS

Edge = pd.concat([U_2_T, T_2_M, T_2_U, M_2_T])

In [61]:
Edge.head()

,src,dst
0,0,1992
1,0,1993
2,0,1994
3,0,1995
4,0,1996


In [64]:
# ### Now the feature data
# Feature data needs to be is sorted in order, where the row index corresponds to the node ID
#
# The data is comprised of three sets of features
# * Transactions
# * Merchants
# * Users

# #### To get feature vectors of Transaction nodes, transform the training data using pre-fitted transformer

transaction_feature_df = pd.DataFrame(
    transformer.transform(data_gnn[predictor_columns]),
    columns=columns_of_transformed_data,
).astype(type_mapping)

transaction_feature_df[COL_FRAUD] = data_gnn[COL_FRAUD]

data_merchant = data_gnn[[COL_MERCHANT, COL_MCC, COL_CARD]].drop_duplicates(
    subset=[COL_MERCHANT]
)
data_merchant[COL_MERCHANT_ID] = data_merchant[COL_MERCHANT].map(
    merchant_name_to_id
)
data_merchant_sorted = data_merchant.sort_values(by=COL_MERCHANT_ID)

data_user = data_gnn[[COL_MERCHANT, COL_MCC, COL_CARD]].drop_duplicates(
    subset=[COL_CARD]
)
data_user[COL_USER_ID] = data_user[COL_CARD].map(id_to_consecutive_id)
data_user_sorted = data_user.sort_values(by=COL_USER_ID)

user_feature_columns = []
mx_feature_columns = []
for c in columns_of_transformed_id_data:
    if c.startswith("Card"):
        user_feature_columns.append(c)
    else:
        mx_feature_columns.append(c)

preprocessed_merchant_data = pd.DataFrame(
    id_transformer.transform(data_merchant_sorted[MERCHANT_AND_USER_COLS]),
    columns=columns_of_transformed_id_data,
)[mx_feature_columns]

preprocessed_user_data = pd.DataFrame(
    id_transformer.transform(data_user_sorted[MERCHANT_AND_USER_COLS]),
    columns=columns_of_transformed_id_data,
)[user_feature_columns]

# Node feature matrix

U = preprocessed_user_data.values
M = preprocessed_merchant_data.values
T = transaction_feature_df[columns_of_transformed_data].values

combined_cols = (
    user_feature_columns + mx_feature_columns + columns_of_transformed_data
)

node_feature_df = pd.DataFrame(block_diag(U, M, T), columns=combined_cols)


In [65]:
node_feature_df

,Card_0,Card_1,Card_2,Card_3,Card_4,Merchant_0,Merchant_1,Merchant_2,Merchant_3,Merchant_4,...,Zip_5,Zip_6,Zip_7,Zip_8,Zip_9,Zip_10,Chip_Chip Transaction,Chip_Online Transaction,Chip_Swipe Transaction,Amount
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
2,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
4,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88556,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-7.103929
88557,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.177819
88558,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.618345
88559,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.124447


In [68]:
transaction_feature_df

,Errors_0,Errors_1,Errors_2,Errors_3,City_0,City_1,City_2,City_3,City_4,City_5,...,Zip_6,Zip_7,Zip_8,Zip_9,Zip_10,Chip_Chip Transaction,Chip_Online Transaction,Chip_Swipe Transaction,Amount,Fraud
0,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,1.171383,0
1,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,-0.110768,0
2,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0.986992,0
3,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,1.102454,0
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0.777390,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86564,0,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,-7.103929,0
86565,0,0,0,1,0,0,0,1,1,1,...,1,1,0,0,1,0,0,1,0.177819,0
86566,0,0,0,1,0,0,0,0,1,1,...,1,0,1,1,1,0,0,1,0.618345,0
86567,0,0,0,1,0,0,0,0,1,1,...,1,0,1,1,0,0,0,1,0.124447,0
